In [2]:
library(ggplot2)
library(data.table)

## Accuracy SNV phasing in IZ

In [3]:
info_mixed_ancestry = read.table('/srv/home/mlef0011/Phasomix/output/local_ancestry/prediction/high_coverage/IS_corrected/input_path_F1_like_mono_x_admixed_gnomix_PCAWG_POP_no_AMR.tsv', header=T, sep = "\t")
samples = info_mixed_ancestry$sample
quant_path = "/srv/home/mlef0011/Phasomix/output/quantification/"

accuracy_IZ_vec_true = c()
accuracy_IZ_vec_sum = c()
accuracy = c()
sum_tot = 0
true_tot = 0
for (sample  in samples){
    file = paste0(quant_path,sample,"/SNV_quantification_",sample,".txt")
    if(file.exists(file)){
        
        SNVs <- fread(file, data.table=FALSE)
        SNVs_IZ = SNVs[SNVs$minor_cn == 0,]
        SNVs_IZ <- na.omit(SNVs_IZ)
        table_IZ = table(SNVs_IZ$gain_copy)
        accuracy_IZ_vec_true = c(accuracy_IZ_vec_true,table_IZ["TRUE"])
        accuracy_IZ_vec_sum = c(accuracy_IZ_vec_sum,sum(table_IZ))
        accuracy = accuracy = c(accuracy,table_IZ["TRUE"]/sum(table_IZ))
        sum_tot = sum_tot + sum(table_IZ)
        true_tot = true_tot + table_IZ["TRUE"]
    }
}
cat("Accuracy = ", round(sum(accuracy_IZ_vec_true,na.rm=TRUE)/sum(accuracy_IZ_vec_sum,na.rm=TRUE)*100,2),"%")

Accuracy =  95.9 %

## Prop phased SNVs

In [16]:
info_ma = read.table('/srv/home/mlef0011/Phasomix/output/local_ancestry/prediction/high_coverage/IS_corrected/input_path_F1_like_mono_x_admixed_gnomix_PCAWG_POP_no_AMR.tsv', header=T, sep = "\t")

info_ma$ancestry <- sapply(1:nrow(info_ma), function(i) {
   anc <- unique(sort(unlist(strsplit(gsub("[{}]", "", c(info_ma[i, "hapA"], info_ma[i, "hapB"])), ","))))
   paste(anc, collapse = "-")
})

prop_phased_SNV <- do.call(rbind, lapply(1:nrow(info_ma), function(i) {
    sample <- info_ma[i, "sample"]
    SNV_path <- info_ma[i, "input_SNV"]
    SNV_phased_path <- paste0("/srv/home/mlef0011/Phasomix/output/quantification/", sample, "/SNV_phasing_mpileup_", sample, ".txt")
    n_hz_path <- paste0("/srv/home/mlef0011/Phasomix/output/quantification/", sample, "/SNPhz_pos_", sample, ".txt")

    if (file.exists(SNV_path) & file.exists(SNV_phased_path) & file.exists(n_hz_path)) {
        n_hz_snps <- as.integer(system(paste("wc -l <", n_hz_path, "| cut -d' ' -f1"), intern = TRUE))
        SNV <- fread(SNV_path, header = TRUE, sep = "\t", data.table = FALSE)
        SNV = SNV[SNV$`#CHROM`%in% 1:22,]
        SNV = SNV[SNV$FILTER == ".",]
        SNV_phased <- fread(SNV_phased_path, header = TRUE, sep = "\t", data.table = FALSE)
        data.frame(sample = sample, prop_phased_SNV = nrow(SNV_phased) / nrow(SNV) * 100, n_hz_SNPs = n_hz_snps)
    } else {
        NULL
    }
}))

In [17]:
prop_phased_SNV

sample,prop_phased_SNV,n_hz_SNPs
<chr>,<dbl>,<int>
09537dce-c797-4b60-962a-d4c3cd6ab00a,39.31452,2046616
0bfd1043-8177-e3e4-e050-11ac0c4860c5,34.19676,2051821
0bfd1068-3fca-a95b-e050-11ac0c4860c3,36.11814,2051821
0bfd1068-3fcd-a95b-e050-11ac0c4860c3,36.75418,2051821
0ef92ff8-829f-425a-91d8-c594b6e22a2b,24.46168,1518556
10ad692b-4c3d-42de-9b5e-4968441388b3,44.84211,2033036
1447c8cb-25d4-4092-8919-4df08f898d2d,25.10232,1504445
1cd0acf2-3116-4dfa-a063-0a435b9f6da3,29.51470,1512948
418e916b-7a4e-4fab-8616-15dcec4d79f8,40.15361,2044207


In [18]:
summary(prop_phased_SNV$prop_phased_SNV)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  24.46   29.05   34.20   32.77   36.05   44.84 

In [7]:
plot_SNV_phasing = ggplot(prop_phased_SNV, aes(x = prop_phased_SNV, y = n_hz_SNPs, color = ancestry)) +
  geom_point(alpha = 0.9, size = 3.5) +
  scale_color_brewer(palette = "Set1") +
  scale_y_continuous(labels = scales::comma) +
  labs(
    x = "% Phased SNVs",
    y = "Number of heterozygous SNPs",
    color = "Ancestry"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    axis.title = element_text(face = "bold", size = 14),
    axis.text = element_text(color = "black", size = 12),
    legend.position = "right",
    legend.title = element_text(face = "bold")
  )

ggsave("/srv/home/mlef0011/Phasomix/output/figures/accuracy_SNV_phasing.png", plot_SNV_phasing,
       width = 7, height = 5, dpi = 300)

plot_SNV_phasing

ERROR: Error: object 'df_plot' not found
